# 5G Network Intelligence — Multisource Representation Pretraining

## Research question and hypothesis

Can external radio and IP-traffic histories improve original-beam forecasting after fine-tuning? More rows do not imply better PRB predictions. Compare target-only, radio, CESNET, and radio+CESNET; retain negative transfer results.


## Temporal protocol and metrics

Original fit/stop/score cutoffs are 335/383/443 and 395/443/503. Inner1 is exploratory selection; inner2 confirms unchanged choices. Previously reused development data cannot establish unseen generalization. Validation 504–671 and final holdout are excluded. Report original-unit DLPRB MAE, WAPE = sum absolute error / sum absolute actual, coverage and zero false-positive rate for +1/+6/+24 hours across all original beams.


In [ ]:
# Review mode never fits, fine-tunes or calibrates on Run All.
RUN_TRAINING = False


## Configuration

The pilot freezes a univariate 64-unit residual MLP, seed 17, five external epochs with 65,536 windows each, and matched target fine-tuning budgets. These are pilot choices, not a claim of optimal architecture.


In [ ]:
from pathlib import Path
import yaml
from IPython.display import display
import pandas as pd

PROJECT = Path.cwd()  # Extract the verified source package here first.
config_path = PROJECT / 'configs/multisource_neural_pilot.yaml'
if config_path.exists():
    config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
    display(pd.DataFrame(config['folds']))
else:
    print('Source package not attached; review the protocol text without execution.')


## Data provenance and preprocessing

GOTS-F is the target collection. NR1800/NR2100/NR3500 belong to one radio collection; CESNET-TimeSeries24 is another collection. CESNET hourly bytes are not PRB labels. Only audited 80% chronological external prefixes are used. Global per-source training-prefix mean/std scaling is distinct from per-entity normalization. Entities affect sampling only. Windows never cross gaps; source NPZ checksums are verified before use.


In [ ]:
# Inspect actual input-check evidence; no synthetic model fitting.
import json
check = PROJECT / 'reports/multisource_neural_input_check.json'
if check.exists():
    evidence = json.loads(check.read_text(encoding='utf-8'))
    display(pd.DataFrame(evidence['source_provenance']))
    print('Collection-balanced domain weights:', evidence['source_weights'])


## Model and training controls

A causal seven-day seasonal mean plus a shared nonlinear residual predicts three horizons. External pretraining transfers the last fixed epoch, never the best in-prefix diagnostic loss. Target stopping is separate from scoring. Pool weights are 1/6 per radio domain and 1/2 CESNET, then uniform entity sampling. All training requires hosted CUDA. Colab artifacts and reports must resolve to mounted Drive; Kaggle retains Output. Implementation: `scripts/train_multisource_neural.py` and `src/features/pretraining_batches.py`.


In [ ]:
# Opt-in cloud execution only; Run All in review mode skips this cell.
if RUN_TRAINING:
    from scripts.train_multisource_neural import main
    main()
else:
    print('Review mode: training skipped.')


## Evaluation and debugging

Inspect per-epoch checkpoints and loss logs before interpreting scoring tables. An epoch cap is not completed patience. Compare every source case at each horizon with the matched target-only case, then historical hurdle and MLP controls. Saved forecast replay, coverage, checksum and inference parity verification are required before accepting any gain.


In [ ]:
# Display available evidence only; absent reports mean no measured result.
for case in ('target_only', 'radio', 'cesnet', 'radio_and_cesnet'):
    report = PROJECT / f'reports/multisource_neural_{case}_development.csv'
    if report.exists():
        print(case)
        display(pd.read_csv(report))
    else:
        print(case, ': no completed result available')


## Conclusions and limitations

No improvement is claimed before cloud execution and independent artifact verification. This pilot uses one seed and univariate inputs; traffic-to-PRB transfer can fail. Sampling balances collections but does not prove deployment-domain similarity. No production replacement or WAPE <10% claim follows from input checks.
